# โครงการวิจัย: การพัฒนาแพลตฟอร์มปัญญาประดิษฐ์บนอุปกรณ์พกพาสำหรับจำแนกแมลงศัตรูข้าวและให้คำแนะนำเชิงปฏิบัติแบบเรียลไทม์

สมุดบันทึกนี้ออกแบบมาเพื่อการฝึกสอน ตรวจสอบ และแปลงโมเดลปัญญาประดิษฐ์สำหรับการจำแนกแมลงศัตรูข้าว 22 ชนิดของประเทศไทย เพื่อรองรับการนำไปติดตั้งบนแอปพลิเคชันมือถือ (TensorFlow Lite)

### 📋 สถาปัตยกรรมและเวิร์กโฟลว์ของสมุดบันทึก
1. **ระบบเตรียมความพร้อม**: เชื่อมต่อ Google Drive และตั้งค่าสภาพแวดล้อม (TensorFlow T4 GPU)
2. **พาธสำหรับ Dataset**: ตรวจสอบตำแหน่งโฟลเดอร์ภาพและไฟล์คำแนะนำ
3. **ตรวจสอบจำนวนภาพสองแหล่งข้อมูลใน GBIF API**: ดึงชื่อวิทยาศาสตร์ทั้ง 2 แหล่งข้อมูล (แบบดั้งเดิม vs แบบใหม่) และวิเคราะห์จำนวนรูปภาพถ่ายที่มีอยู่บนฐานข้อมูลสากล GBIF
4. **การทำความสะอาดภาพ**: ตรวจสอบภาพชำรุดและลบไฟล์ภาพที่ไม่สมบูรณ์
5. **การพรีโพรเซสภาพ**: ปรับขนาดภาพด้วยแนวทาง **Letterbox Resizing with Padding** ขนาด 224x224 พิกเซล เพื่อรักษาสันฐานวิทยาแมลง
6. **ท่อส่งข้อมูลการเทรน**: ทำการแบ่งชุดข้อมูล (Train 80% / Val 20%) และเพิ่มความทนทานโมเดลด้วย **Data Augmentation**
7. **การฝึกสอนและเปรียบเทียบโมเดล (Two-Phase Transfer Learning)**:
   - **โมเดลที่เปรียบเทียบ**: `MobileNetV2`, `EfficientNetB0`, `ResNet50V2`
   - **เฟสที่ 1 (Feature Extraction)**: ตรึงน้ำหนักของ Base Network และฝึกสอนเฉพาะชั้นจำแนก
   - **เฟสที่ 2 (Fine-Tuning)**: ปลดล็อกเลเยอร์ระดับสูง (Top Layers) ของแบบจำลองและจูนรายละเอียดด้วยอัตราการเรียนรู้ระดับต่ำสุด
8. **การประเมินและเปรียบเทียบ**: พล็อตและวิเคราะห์กราฟความถูกต้อง (Accuracy) และความสูญเสีย (Loss)
9. **การแปลงน้ำหนักโมเดล**: ส่งออกโมเดลที่ดีที่สุดเป็นไฟล์ `.tflite` สำหรับติดตั้งบนอุปกรณ์พกพา
10. **ระบบผู้เชี่ยวชาญ (Inference with Real-time Advice)**: การจำแนกภาพแมลงเดี่ยวและเชื่อมโยงคำแนะนำเกษตรจากไฟล์ `recommendations.json` ทันที

## 🛠️ ส่วนที่ 1: การโหลดไลบรารีที่จำเป็นและการตั้งค่าอุปกรณ์ประมวลผล

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import urllib.request
import urllib.parse
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow Version:", tf.__version__)
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print('WARNING: GPU device not found. Please change runtime to GPU (T4).')
else:
    print('Success: Found GPU at:', device_name)

## 📂 ส่วนที่ 2: เชื่อมต่อ Google Drive และตั้งค่าพาธสำหรับ Dataset

In [14]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # พาธมาตรฐานเมื่อนำโฟลเดอร์ไปวางใน Google Drive
    DATASET_DIR = '/content/drive/MyDrive/Colab Notebooks/insect/dataset/images'
    RECOMMENDATIONS_PATH = '/content/drive/MyDrive/Colab Notebooks/insect/data/recommendations.json'
except:
    print("Running on local machine or standard workspace.")
    DATASET_DIR = 'dataset/images'
    RECOMMENDATIONS_PATH = 'data/recommendations.json'

print("Checking Dataset location:", DATASET_DIR)
if os.path.exists(DATASET_DIR):
    classes = sorted(os.listdir(DATASET_DIR))
    print(f"Found {len(classes)} classes of rice insect pests:")
    for idx, cls in enumerate(classes):
        count = len(os.listdir(os.path.join(DATASET_DIR, cls)))
        print(f"{idx+1}. {cls}: {count} images")
else:
    print("Error: Dataset directory not found. Please upload dataset/images to your workspace or Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking Dataset location: /content/drive/MyDrive/Colab Notebooks/insect/dataset/images
Error: Dataset directory not found. Please upload dataset/images to your workspace or Google Drive.


## 🌐 ส่วนที่ 3: การตรวจสอบและเปรียบเทียบจำนวนภาพของทั้งสองแหล่งข้อมูลบนคลังข้อมูลโลก GBIF (Query & Compare GBIF Counts)
ส่วนนี้จะทำการอ่านไฟล์คำแนะนำภาษาไทยและดึงชื่อวิทยาศาสตร์ (ทั้งจากแหล่งที่ 1 ดั้งเดิม และแหล่งที่ 2 เพิ่มเติมใหม่) ของแมลงศัตรูข้าวทั้ง 22 ชนิด จากนั้นส่งสืบค้นไปยัง GBIF Occurrence API โดยใช้ฟังก์ชันในการตัดแต่งข้อมูลชื่อ (Author/Parentheses name filter) เพื่อดึงและแสดงสถิติจำนวนภาพเคียงข้างกัน

In [16]:
def clean_sci_name(name):
    if not name:
        return ''
    for sep in ['/', 'or', ';']:
        if sep in name:
            name = name.split(sep)[0]
    name = name.strip()
    words = name.split()
    if len(words) >= 2:
        if words[1] in ['sp.', 'spp.']:
            return words[0]
        # กรองตัวอักษรพิเศษที่ไม่ใช่ภาษาอังกฤษและเครื่องหมายขีดคั่นออก
        w1 = ''.join(c for c in words[0] if c.isalpha() or c=='-')
        w2 = ''.join(c for c in words[1] if c.isalpha() or c=='-')
        return f"{w1} {w2}"
    return name

print("Reading recommendations.json from:", RECOMMENDATIONS_PATH)
if os.path.exists(RECOMMENDATIONS_PATH):
    with open(RECOMMENDATIONS_PATH, 'r', encoding='utf-8') as f:
        pest_db = json.load(f)
    
    gbif_results = []
    print("Querying GBIF API for scientific names (mediaType=StillImage)...\n")
    
    for thai_name, info in pest_db.items():
        name1 = info.get('scientific_name', '')
        name2 = info.get('scientific_name_2', '')
        
        # clean_name1 = clean_sci_name(name1)
        # clean_name2 = clean_sci_name(name2)
        clean_name1 = (name1)
        clean_name2 = (name2)
        
        # คิวรีชื่อวิทยาศาสตร์ 1
        c1 = 0
        if clean_name1:
            url1 = f"https://api.gbif.org/v1/occurrence/search?scientificName={urllib.parse.quote(clean_name1)}&mediaType=StillImage&limit=0"
            try:
                with urllib.request.urlopen(url1, timeout=10) as r:
                    c1 = json.loads(r.read().decode('utf-8')).get('count', 0)
            except:
                c1 = "Error"
        
        # คิวรีชื่อวิทยาศาสตร์ 2
        c2 = 0
        if clean_name2:
            url2 = f"https://api.gbif.org/v1/occurrence/search?scientificName={urllib.parse.quote(clean_name2)}&mediaType=StillImage&limit=0"
            try:
                with urllib.request.urlopen(url2, timeout=10) as r:
                    c2 = json.loads(r.read().decode('utf-8')).get('count', 0)
            except:
                c2 = "Error"
        else:
            c2 = "-"
            
        gbif_results.append({
            "ชื่อภาษาไทย": thai_name,
            "ชื่อวิทยาศาสตร์ 1 (เดิม)": name1,
            "จำนวนภาพ 1 (GBIF)": c1,
            "ชื่อวิทยาศาสตร์ 2 (ใหม่)": name2,
            "จำนวนภาพ 2 (GBIF)": c2
        })
        print(f"- {thai_name}: {clean_name1} ({c1} ภาพ) vs {clean_name2} ({c2} ภาพ)")
        
    # แปลงเป็น DataFrame
    df_gbif = pd.DataFrame(gbif_results)
    print("\n=== ตารางเปรียบเทียบจำนวนภาพถ่ายระหว่างชื่อดั้งเดิมและแหล่งข้อมูลใหม่บน GBIF ===")
    display(df_gbif)
else:
    print(f"Error: recommendations.json not found at {RECOMMENDATIONS_PATH}")

Reading recommendations.json from: /content/drive/MyDrive/Colab Notebooks/insect/data/recommendations.json
Querying GBIF API for scientific names (mediaType=StillImage)...

- ด้วงงวงกินรากข้าว: Echinocnemus oryzae (0 ภาพ) vs Hydronomidius molitor Faust (0 ภาพ)
- ด้วงดำ: Heteronychus lioderes (0 ภาพ) vs Heteronychus lioderes Redtenbacher (0 ภาพ)
- มวนง่าม: Cletus punctiger (206 ภาพ) vs Tetroda denticulifera (Berg) (0 ภาพ)
- หนอนกระทู้กล้า: Spodoptera mauritia (2463 ภาพ) vs Spodoptera mauritia (Boisduval) (2463 ภาพ)
- หนอนกระทู้คอรวง: Mythimna separata (837 ภาพ) vs Mythimna separata (Walker) (837 ภาพ)
- หนอนกอข้าวสีครีม: Scirpophaga innotata (10 ภาพ) vs Scirpophaga incertulas (Walker) (439 ภาพ)
- หนอนกอสีชมพู: Sesamia inferens (95 ภาพ) vs Sesamia inferens (Walker) (95 ภาพ)
- หนอนกอแถบลายสีม่วง: Chilo polychrysus (0 ภาพ) vs Chilo polychrysus (Meyrick) (0 ภาพ)
- หนอนกอแถบลายเล็ก: Chilo suppressalis (57 ภาพ) vs Chilo suppressalis (Walker) (57 ภาพ)
- หนอนปลอกข้าว: Parapoynx fluctuosalis (399

,ชื่อภาษาไทย,ชื่อวิทยาศาสตร์ 1 (เดิม),จำนวนภาพ 1 (GBIF),ชื่อวิทยาศาสตร์ 2 (ใหม่),จำนวนภาพ 2 (GBIF)
0,ด้วงงวงกินรากข้าว,Echinocnemus oryzae,0,Hydronomidius molitor Faust,0
1,ด้วงดำ,Heteronychus lioderes,0,Heteronychus lioderes Redtenbacher,0
2,มวนง่าม,Cletus punctiger,206,Tetroda denticulifera (Berg),0
3,หนอนกระทู้กล้า,Spodoptera mauritia,2463,Spodoptera mauritia (Boisduval),2463
4,หนอนกระทู้คอรวง,Mythimna separata,837,Mythimna separata (Walker),837
5,หนอนกอข้าวสีครีม,Scirpophaga innotata,10,Scirpophaga incertulas (Walker),439
6,หนอนกอสีชมพู,Sesamia inferens,95,Sesamia inferens (Walker),95
7,หนอนกอแถบลายสีม่วง,Chilo polychrysus,0,Chilo polychrysus (Meyrick),0
8,หนอนกอแถบลายเล็ก,Chilo suppressalis,57,Chilo suppressalis (Walker),57
9,หนอนปลอกข้าว,Parapoynx fluctuosalis,399,Nymphula depunctalis Guenée,0


## 🧹 ส่วนที่ 4: ตรวจสอบและทำความสะอาดภาพชำรุด (Data Integrity Check)
ฟังก์ชันนี้จะช่วยตรวจสอบว่ามีรูปภาพใดที่ไม่สามารถเปิดอ่านได้ปกติ หรือมีขนาดไฟล์เป็น 0 เพื่อป้องกันไม่ให้ท่อเทรนพังขณะประมวลผล

In [ ]:
def check_and_clean_images(directory):
    corrupted_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                file_path = os.path.join(root, file)
                try:
                    img = Image.open(file_path)
                    img.verify() # ตรวจสอบความถูกต้องทางโครงสร้างของไฟล์
                except (IOError, SyntaxError) as e:
                    print(f"Corrupted image detected: {file_path} - Error: {e}")
                    corrupted_files.append(file_path)
    return corrupted_files

if os.path.exists(DATASET_DIR):
    print("Scanning dataset for corrupted images...")
    corrupted = check_and_clean_images(DATASET_DIR)
    if len(corrupted) == 0:
        print("All images verified. No corrupted files found!")
    else:
        print(f"Found {len(corrupted)} corrupted files.")
        # หากผู้ใช้ต้องการลบ ให้ปลดคอมเมนต์โค้ดบรรทัดด้านล่างเพื่อเคลียร์ภาพเสียออก
        # for path in corrupted:
        #     os.remove(path)
        #     print(f"Deleted corrupted file: {path}")

## 📐 ส่วนที่ 5: การประมวลผลขนาดภาพโดยรักษาอัตราส่วน (Letterbox Resizing with Padding)
การย่อภาพแบบปกติอาจทำให้ลักษณะทางสัณฐานวิทยาของแมลง เช่น ความกว้างลำตัว ขา หรือหนวดเกิดการบิดเบี้ยวได้ เราจึงใช้วิธี **Letterbox** โดยการปรับให้ความกว้างหรือความสูงเท่ากับขนาดเป้าหมายก่อนแล้วเติมขอบว่างส่วนที่ขาดด้วยสีขาว/ดำ

In [ ]:
def letterbox_image(image_path, target_size=(224, 224), padding_color=(255, 255, 255)):
    """
    ฟังก์ชันปรับขนาดภาพโดยรักษาอัตราส่วนและเติมขอบว่าง (Letterbox with Padding)
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    target_w, target_h = target_size

    # คำนวณสัดส่วนสเกลที่ต้องปรับ
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    # ปรับขนาดภาพแบบรักษาอัตราส่วน
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # สร้างกรอบภาพขนาดเป้าหมายพร้อมเติมพื้นหลังสีที่กำหนด
    padded = np.full((target_h, target_w, 3), padding_color, dtype=np.uint8)

    # คำนวณหาพิกัดกึ่งกลางเพื่อเอาภาพสเกลวางตรงกลาง
    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    # วางภาพทับลงในกรอบพิกัดกึ่งกลาง
    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return padded

# แสดงภาพตัวอย่างการทำ Letterbox
if os.path.exists(DATASET_DIR):
    first_class = classes[0]
    class_dir = os.path.join(DATASET_DIR, first_class)
    first_image = os.path.join(class_dir, os.listdir(class_dir)[0])
    
    original_img = Image.open(first_image)
    letterboxed_img = letterbox_image(first_image, target_size=(224, 224))
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title(f"Original ({original_img.size[0]}x{original_img.size[1]})")
    plt.imshow(original_img)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Letterboxed with Padding (224x224)")
    plt.imshow(letterboxed_img)
    plt.axis('off')
    plt.show()

## 🔄 ส่วนที่ 6: การจัดแบ่งชุดข้อมูลการเทรนและ Data Augmentation
เราจะประยุกต์ใช้ ImageDataGenerator ร่วมกับการหมุนสุ่ม พลิกภาพ และการสเกลขนาดสำหรับ Train และ Validation ในอัตราส่วน 80:20 เพื่อลดโอกาสเกิด Overfitting

In [ ]:
BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)

# ใช้เทคนิคการขยายข้อมูล (Data Augmentation) สำหรับชุดฝึกสอน
train_datagen = ImageDataGenerator(
    rescale=1./255,               # ทำการ Normalization ปรับช่วงสีจาก 0-255 เป็น 0-1
    rotation_range=20,            # สุ่มหมุนภาพไม่เกิน 20 องศา
    width_shift_range=0.1,        # สุ่มเลื่อนด้านข้าง
    height_shift_range=0.1,       # สุ่มเลื่อนแนวตั้ง
    zoom_range=0.1,               # สุ่มซูม
    brightness_range=[0.9, 1.1],  # สุ่มปรับระดับความสว่าง
    horizontal_flip=True,         # สุ่มพลิกภาพแนวตั้งหรือแนวนอน
    validation_split=0.2          # กำหนดสัดส่วนแบ่งทำชุดตรวจสอบ 20%
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# โหลดข้อมูล Train (80%)
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42,
    shuffle=True
)

# โหลดข้อมูล Validation (20%)
val_generator = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42,
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print("Detected number of classes:", num_classes)
print("Class Map:", train_generator.class_indices)

## 🧠 ส่วนที่ 7: การสร้าง ฝึกสอน และเปรียบเทียบสถาปัตยกรรมโมเดล Deep Learning
เราทำการเปรียบเทียบโมเดล 3 โมเดล ได้แก่ `MobileNetV2` (น้ำหนักเบา รันบนมือถือดีเยี่ยม), `EfficientNetB0` (ค่าความแม่นยำคุ้มค่าสเกลพารามิเตอร์) และ `ResNet50V2` (โครงสร้างลึกทรงประสิทธิภาพ)

In [ ]:
def build_transfer_learning_model(model_name, input_shape=(224, 224, 3), num_classes=22):
    """
    สร้างแบบจำลองโดยตรึงน้ำหนักโครงสร้างหลักที่ใช้ ImageNet และสวมหัว Classification เพิ่มเติม
    """
    if model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_name == "EfficientNetB0":
        base_model = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_name == "ResNet50V2":
        base_model = tf.keras.applications.ResNet50V2(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError("Unknown model name")
        
    # ตรึงโมเดลหลัก (แช่แข็งค่าน้ำหนักฟีเจอร์)
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

### 7.1 เฟสแรก: การเรียนรู้สกัดฟีเจอร์ (Feature Extraction Phase)
ในส่วนนี้เราจะล็อคโครงสร้างด้านล่างไว้และเทรนเฉพาะ Classifier ด้านบนเป็นเวลา 20-30 epochs ด้วยอัตราการเรียนรู้เริ่มต้น $10^{-4}$

In [ ]:
histories = {}
trained_models = {}

selected_models = ["MobileNetV2", "EfficientNetB0", "ResNet50V2"]

for name in selected_models:
    print(f"\n=== Start Feature Extraction for {name} ===")
    model, base_model = build_transfer_learning_model(name, num_classes=num_classes)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4)
    ]
    
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=callbacks
    )
    
    histories[name] = history.history
    trained_models[name] = (model, base_model)

### 7.2 เฟสสอง: การปรับแต่งเชิงลึก (Fine-Tuning Phase)
ปลดล็อกโมดูลบล็อกท้ายๆ ของ Base model และฝึกสอนพร้อมกันด้วยอัตราการเรียนรู้ระดับต่ำมาก $10^{-5}$

In [ ]:
fine_tune_histories = {}

for name in selected_models:
    print(f"\n=== Start Fine-Tuning for {name} ===")
    model, base_model = trained_models[name]
    
    base_model.trainable = True
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)
    ]
    
    history_ft = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=callbacks
    )
    
    fine_tune_histories[name] = history_ft.history

## 📊 ส่วนที่ 8: พล็อตและเปรียบเทียบผลลัพธ์การฝึกสอน (Evaluation & Visualization)

In [ ]:
plt.figure(figsize=(15, 10))

for idx, name in enumerate(selected_models):
    acc_total = histories[name]['accuracy'] + fine_tune_histories[name]['accuracy']
    val_acc_total = histories[name]['val_accuracy'] + fine_tune_histories[name]['val_accuracy']
    
    plt.subplot(2, 2, idx+1)
    plt.plot(acc_total, label='Train Acc', color='teal', linestyle='-')
    plt.plot(val_acc_total, label='Val Acc', color='coral', linestyle='--')
    plt.axvline(x=len(histories[name]['accuracy']), color='gray', linestyle=':', label='Fine-Tuning Start')
    plt.title(f"{name} Performance Curves")
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

## 💾 ส่วนที่ 9: บันทึกและแปลงโมเดลที่ดีที่สุดเป็น TensorFlow Lite (.tflite)
โมเดลจำแนกแมลงที่ฝึกสอนสมบูรณ์แล้วจะถูกเซฟเก็บและทำการแปลงเป็น TFLite เพื่อให้สามารถโหลดใช้งานอย่างรวดเร็วและใช้พลังงานต่ำในแอปพลิเคชันพกพาของเกษตรกร

In [ ]:
best_model_name = "EfficientNetB0"
model_to_export, _ = trained_models[best_model_name]

keras_model_path = 'rice_insect_best_model.h5'
model_to_export.save(keras_model_path)
print(f"Saved best model {best_model_name} to {keras_model_path}")

converter = tf.lite.TFLiteConverter.from_keras_model(model_to_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = 'rice_insect_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Success: TFLite model generated and saved to {tflite_path}")
print("Size of Keras H5 model:", os.path.getsize(keras_model_path) / (1024*1024), "MB")
print("Size of TFLite model (optimized):", os.path.getsize(tflite_path) / (1024*1024), "MB")

## 🚜 ส่วนที่ 10: ระบบทดสอบการจำแนกและดึงคำแนะนำเชิงปฏิบัติการกำจัดแมลงศัตรูข้าว (Real-time AI Inference & Advice)
จำลองการทำงานบนแอปพลิเคชันพกพาเมื่อผู้ใช้งานถ่ายรูปและอัปโหลดระบบจำแนกจะให้คำแนะนำในการจัดการตามปัญญาประดิษฐ์และคลังภูมิปัญญาของไทยทันที

In [ ]:
def get_advice_for_pest(pest_name_thai, rec_json_path=RECOMMENDATIONS_PATH):
    """
    ฟังก์ชันสืบค้นข้อแนะนำตามคลาสแมลงภาษาไทยจาก recommendations.json
    """
    if not os.path.exists(rec_json_path):
        return "คำแนะนำ: ไม่พบฐานข้อมูลคำแนะนำเกษตรกร กรุณาจัดเตรียมไฟล์ recommendations.json"
        
    with open(rec_json_path, 'r', encoding='utf-8') as f:
        database = json.load(f)
        
    if pest_name_thai in database:
        info = database[pest_name_thai]
        name_2_text = f" / {info['scientific_name_2']}" if info.get('scientific_name_2') else ""
        rec_text = "\n- ".join(info['recommendations'])
        return f"📌 ชื่อภาษาอังกฤษ: {info['english_name']}\n🔬 ชื่อวิทยาศาสตร์: {info['scientific_name']}{name_2_text}\n⚠ รายละเอียด: {info['description']}\n\n✅ คำแนะนำเชิงปฏิบัติ:\n- {rec_text}"
    else:
        return "ไม่พบคู่มือแนะนำเฉพาะเจาะจงสำหรับแมลงชนิดนี้ในฐานข้อมูล"

def predict_and_advise(image_path, model, class_indices, target_size=(224, 224)):
    """
    ทำนายภาพแมลงศัตรูข้าวและแสดงผลคำแนะนำคู่กัน
    """
    processed_img = letterbox_image(image_path, target_size=target_size)
    if processed_img is None:
        print("Error loading image")
        return
        
    img_input = np.expand_dims(processed_img / 255.0, axis=0)
    
    preds = model.predict(img_input)
    pred_idx = np.argmax(preds[0])
    confidence = preds[0][pred_idx] * 100
    
    inverse_map = {v: k for k, v in class_indices.items()}
    pred_class_thai = inverse_map[pred_idx]
    
    plt.figure(figsize=(6, 6))
    plt.imshow(processed_img)
    plt.title(f"Prediction: {pred_class_thai} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()
    
    print("=================== รายงานและคำแนะนำเชิงปฏิบัติการเกษตร ===================")
    print(f"พบศัตรูข้าวชนิด: {pred_class_thai} (ความมั่นใจ {confidence:.2f}%)")
    advice = get_advice_for_pest(pred_class_thai)
    print(advice)
    print("=======================================================================")

if os.path.exists(DATASET_DIR):
    first_class = classes[0]
    class_dir = os.path.join(DATASET_DIR, first_class)
    test_image = os.path.join(class_dir, os.listdir(class_dir)[0])
    
    model_instance, _ = trained_models["EfficientNetB0"]
    predict_and_advise(test_image, model_instance, train_generator.class_indices)